# `@dataclass`

A class that mainly holds data always needs the same code: an `__init__`
that stores each field, a `__repr__`, an `__eq__`. `@dataclass` writes it
from the list of fields.

In [1]:
from dataclasses import dataclass


@dataclass
class Venue:
    name: str
    capacity: int
    address: str = ""


kiosque = Venue("Le Kiosque", 120)
print(kiosque)
print(kiosque == Venue("Le Kiosque", 120))
print(kiosque.capacity)

Venue(name='Le Kiosque', capacity=120, address='')
True
120


It is exactly as if the class had been written:

```python
class Venue:
    def __init__(self, name: str, capacity: int, address: str = "") -> None:
        self.name = name
        self.capacity = capacity
        self.address = address

    def __repr__(self) -> str:
        return f"Venue(name={self.name!r}, capacity={self.capacity!r}, address={self.address!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Venue):
            return NotImplemented
        return (self.name, self.capacity, self.address) == (other.name, other.capacity, other.address)
```

## Checking the values: `__post_init__`

In [2]:
@dataclass
class Venue:
    name: str
    capacity: int

    def __post_init__(self):
        self.name = self.name.strip()
        if self.capacity < 1:
            raise ValueError(f"A venue needs at least one seat, got {self.capacity}.")


print(Venue("  Le Kiosque ", 120))

Venue(name='Le Kiosque', capacity=120)


In [3]:
Venue("Nowhere", 0)

ValueError: A venue needs at least one seat, got 0.

`__post_init__` runs at the end of the generated `__init__`. The annotations (`capacity: int`) are not checked by Python: checks go here.

## A list as default value

In [4]:
@dataclass
class Lineup:
    host: str
    acts: list[str] = []

ValueError: mutable default <class 'list'> for field acts is not allowed: use default_factory

In [5]:
from dataclasses import field


@dataclass
class Lineup:
    host: str
    acts: list[str] = field(default_factory=list)


gala = Lineup("Paul Kerbrat")
club = Lineup("Tom Vasseur")
gala.acts.append("Nora Lebrun")
print(gala)
print(club)

Lineup(host='Paul Kerbrat', acts=['Nora Lebrun'])
Lineup(host='Tom Vasseur', acts=[])


`default_factory=list` creates a new empty list for each lineup.

## `@property`: an attribute computed at each read

In [6]:
from datetime import date, datetime, timedelta


@dataclass
class Show:
    title: str
    start: datetime
    duration: int

    @property
    def end(self) -> datetime:
        return self.start + timedelta(minutes=self.duration)

    @property
    def festival_day(self) -> date:
        return (self.start - timedelta(hours=6)).date()


late = Show("Late Laughs", datetime(2027, 6, 12, 0, 30), 60)
print(late.end)
print(late.festival_day)

2027-06-12 01:30:00
2027-06-11


In [7]:
late.duration = 90
print(late.end)

2027-06-12 02:00:00


In [8]:
late.end = datetime(2027, 6, 12, 2, 0)

AttributeError: property 'end' of 'Show' object has no setter

A property is a method read without parentheses. It is computed again at
each read, so it follows the changes of the fields, and it cannot be
assigned.

| What | Written as |
|---|---|
| data given when creating the object | field |
| value computed from the fields, no parameter | `@property` |
| action, or question with parameters | method |

## Replacing a generated method

In [9]:
@dataclass
class Artist:
    name: str
    bio: str = ""

    def __eq__(self, other):
        if not isinstance(other, Artist):
            return NotImplemented
        return self.name.lower() == other.name.lower()


print(Artist("Nora Lebrun", "Stories.") == Artist("NORA LEBRUN"))

True


A special method written in the class is kept: `@dataclass` only writes the missing ones.